#PATHS AND PARAMETERS


In [1]:
# ============================================================
# 0) PATHS AND PARAMETERS
# ============================================================
import os

# --- BASE MODEL ARTIFACTS ---
BASE_OUT_DIR = "outputs_nn_base"
MODEL_BASE_FULL_PATH = os.path.join(BASE_OUT_DIR, "base_model_full_data.keras")
PREPROCESSOR_PATH = os.path.join(BASE_OUT_DIR, "preprocessor.joblib")
META_PATH = os.path.join(BASE_OUT_DIR, "meta.json")

# --- New dataset ---
NEW_DATA_PATH = "/content/sample_data/UCHW.xlsx"
NEW_SHEET = None
TARGET_NEW = "acute_GvHD_II_III_IV"

# Percentages
TEST_SIZE_GLOBAL = 0.15   # ~15% test
RANDOM_STATE_NEW = 123
# Tuner
MAX_TRIALS_TRANSFER = 10
EPOCHS_TRANSFER = 1000
BATCH_SIZE_TRANSFER = 16

print("Base model:", MODEL_BASE_FULL_PATH)
print("Base preprocessor:", PREPROCESSOR_PATH)
print("Base meta:", META_PATH)
print("New dataset:", NEW_DATA_PATH)


Base model: outputs_nn_base/base_model_full_data.keras
Base preprocessor: outputs_nn_base/preprocessor.joblib
Base meta: outputs_nn_base/meta.json
New dataset: /content/sample_data/UCHW.xlsx


#IMPORTS AND SEEDS


In [2]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.8 MB/s eta 0:00:00


In [3]:
# ============================================================
# 1) IMPORTS Y SEEDS
# ============================================================
import random
import json
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score
)

import joblib
import keras_tuner as kt

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


# LOAD META, PREPROCESSOR AND BASE MODEL


In [4]:
# ============================================================
# 2) LOAD META, PREPROCESADOR Y MODEL BASE
# ============================================================
with open(META_PATH, "r") as f:
    meta_base = json.load(f)

print("Base meta:")
print(json.dumps(meta_base, indent=2))

target_base = meta_base["target"]
cols_base = meta_base["columns_used"]      # includes target
cat_cols_base = meta_base["cat_cols"]
num_cols_base = meta_base["num_cols"]
input_dim_base = meta_base["input_dim_after_preprocessing"]

# Feature columns used by the base model (without the target)
existing_feature_cols = [c for c in cols_base if c != target_base]

print("\nExisting features (base model):", existing_feature_cols)
print("Input dim after base preprocessing:", input_dim_base)

# Load base preprocessor and model
pre_base = joblib.load(PREPROCESSOR_PATH)
base_model = tf.keras.models.load_model(MODEL_BASE_FULL_PATH)

base_model.summary()


Base meta:
{
  "target": "acute_GvHD_II_III_IV",
  "columns_used": [
    "recipient_gender",
    "disease",
    "stem_cell_source",
    "acute_GvHD_II_III_IV"
  ],
  "cat_cols": [
    "recipient_gender",
    "disease",
    "stem_cell_source"
  ],
  "num_cols": [],
  "test_size": 0.15,
  "random_state": 16,
  "best_epoch": 20,
  "best_hyperparameters": {
    "n_pre_layers": 0,
    "pre_units": 32,
    "l2": 0.0001,
    "dropout_pre": 0.0,
    "encoder_units": 8,
    "dropout_enc": 0.2,
    "lr": 0.001
  },
  "n_train": 158,
  "n_val": 29,
  "n_full": 187,
  "input_dim_after_preprocessing": 8,
  "model_path": "outputs_nn_base/base_model_full_data.keras",
  "preprocessor_path": "outputs_nn_base/preprocessor.joblib"
}

Existing features (base model): ['recipient_gender', 'disease', 'stem_cell_source']
Input dim after base preprocessing: 8


Model: "base_gvhd_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout (Dropout)       │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gvhd_head (Dense)               │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 245 (984.00 B)

 Trainable params: 81 (324.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 164 (660.00 B)

#EXTRACT THE BASE MODEL ENCODER


In [5]:
# ============================================================
# 3) EXTRACT THE ENCODER FROM THE BASE MODEL
# ============================================================
encoder_layer = base_model.get_layer("encoder")  # Defined name on build_model()
encoder_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=encoder_layer.output,
    name="gvhd_encoder"
)

encoder_model.summary()


Model: "gvhd_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Dense)                 │ (None, 8)              │            72 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72 (288.00 B)

 Trainable params: 72 (288.00 B)

 Non-trainable params: 0 (0.00 B)

#LOAD NEW DATASET AND DEFINE X_existing / X_new


In [6]:
# ============================================================
# 4) LOAD NEW DATASET Y DEFINIR X_existing / X_new
# ============================================================
if NEW_SHEET is None:
    df_new = pd.read_excel(NEW_DATA_PATH)
else:
    df_new = pd.read_excel(NEW_DATA_PATH, sheet_name=NEW_SHEET)

assert TARGET_NEW in df_new.columns, f"Target column was not found '{TARGET_NEW}' in the new dataset"
# Filter per type of trasplantation
df_new = df_new[df_new["Type_of_transplant"] == 1]
df_new = df_new.dropna()
df_new=df_new.loc[:,['recipient_gender','disease','stem_cell_source','Previous_therapy','Graft_manipulation','Only_Chemotherapy','Degree_of_kinship','Chemotherapy_Irradiation','acute_GvHD_II_III_IV']]
print("Shape new dataset (raw):", df_new.shape)
df_new.head()


Shape new dataset (raw): (126, 9)


,recipient_gender,disease,stem_cell_source,Previous_therapy,Graft_manipulation,Only_Chemotherapy,Degree_of_kinship,Chemotherapy_Irradiation,acute_GvHD_II_III_IV
0,male,Leukemia,peripheral_blood,0,no manipulation,0,UD,0,1
1,male,Leukemia,peripheral_blood,1,no manipulation,0,UD,1,1
3,female,lymphoma,peripheral_blood,1,no manipulation,1,UD,0,1
4,female,Leukemia,peripheral_blood,1,no manipulation,0,Sibiling,1,1
6,female,Leukemia,peripheral_blood,0,no manipulation,0,UD,0,0


#Check 'existing' columns in the new dataset


In [7]:
# --- Check 'existing' columns in the new dataset ---
missing_existing = [c for c in existing_feature_cols if c not in df_new.columns]
if missing_existing:
    print("WARNING: faltan columnas 'existing' in the new dataset:", missing_existing)

existing_cols_in_new = [c for c in existing_feature_cols if c in df_new.columns]

# New columns = all columns that are neither existing nor target
feature_cols_new = [
    c for c in df_new.columns
    if c not in existing_cols_in_new and c != TARGET_NEW
]

print("X_existing columns (in new dataset):", existing_cols_in_new)
print("X_new columns:", feature_cols_new)

# We keep existing + new + target
cols_all = existing_cols_in_new + feature_cols_new + [TARGET_NEW]
df_new = df_new.loc[:, cols_all]

# dropna
df_new = df_new.dropna()
print("New dataset shape after dropna:", df_new.shape)
print("New target distribution:\n", df_new[TARGET_NEW].value_counts())
df_new.head()


X_existing columns (in new dataset): ['recipient_gender', 'disease', 'stem_cell_source']
X_new columns: ['Previous_therapy', 'Graft_manipulation', 'Only_Chemotherapy', 'Degree_of_kinship', 'Chemotherapy_Irradiation']
New dataset shape after dropna: (126, 9)
New target distribution:
 acute_GvHD_II_III_IV
1    75
0    51
Name: count, dtype: int64


,recipient_gender,disease,stem_cell_source,Previous_therapy,Graft_manipulation,Only_Chemotherapy,Degree_of_kinship,Chemotherapy_Irradiation,acute_GvHD_II_III_IV
0,male,Leukemia,peripheral_blood,0,no manipulation,0,UD,0,1
1,male,Leukemia,peripheral_blood,1,no manipulation,0,UD,1,1
3,female,lymphoma,peripheral_blood,1,no manipulation,1,UD,0,1
4,female,Leukemia,peripheral_blood,1,no manipulation,0,Sibiling,1,1
6,female,Leukemia,peripheral_blood,0,no manipulation,0,UD,0,0


#SPLIT INTO TRAIN / VAL / TEST


In [8]:
# ============================================================
# 5) SPLIT EN TRAIN / VAL / TEST
# ============================================================
y_new = df_new[TARGET_NEW].astype(int)
X_new_all = df_new.drop(columns=[TARGET_NEW]).copy()

# 1) Train+Val vs Test
X_trainval_raw, X_test_raw, y_trainval, y_test = train_test_split(
    X_new_all, y_new,
    test_size=TEST_SIZE_GLOBAL,
    random_state=RANDOM_STATE_NEW,
    stratify=y_new
)

print("Train+Val:", X_trainval_raw.shape, "| Test:", X_test_raw.shape)

# 2) Within Train+Val, split Train vs Val so that Val is ~15% of the total
VAL_SIZE = 0.1765  # aprox 0.15 global (0.85 * 0.1765 ≈ 0.15)

X_train_raw, X_val_raw, y_train_new, y_val_new = train_test_split(
    X_trainval_raw, y_trainval,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE_NEW,
    stratify=y_trainval
)

print("Train:", X_train_raw.shape,
      "| Val:", X_val_raw.shape,
      "| Test:", X_test_raw.shape)


Train+Val: (107, 8) | Test: (19, 8)
Train: (88, 8) | Val: (19, 8) | Test: (19, 8)


#Split existing / new on each split


In [9]:
# --- Split existing / new on each split ---
# Train
X_train_existing_raw = X_train_raw[existing_cols_in_new]
X_train_new_raw      = X_train_raw[feature_cols_new]

# Val
X_val_existing_raw = X_val_raw[existing_cols_in_new]
X_val_new_raw      = X_val_raw[feature_cols_new]

# Test
X_test_existing_raw = X_test_raw[existing_cols_in_new]
X_test_new_raw      = X_test_raw[feature_cols_new]

print("X_train_existing_raw:", X_train_existing_raw.shape,
      "| X_train_new_raw:", X_train_new_raw.shape)
print("X_val_existing_raw:", X_val_existing_raw.shape,
      "| X_val_new_raw:", X_val_new_raw.shape)
print("X_test_existing_raw:", X_test_existing_raw.shape,
      "| X_test_new_raw:", X_test_new_raw.shape)


X_train_existing_raw: (88, 3) | X_train_new_raw: (88, 5)
X_val_existing_raw: (19, 3) | X_val_new_raw: (19, 5)
X_test_existing_raw: (19, 3) | X_test_new_raw: (19, 5)


#PREPROCESS X_existing (pre_base) AND X_new (pre_new)


In [10]:
# ============================================================
# 6) PREPROCESAR X_existing (pre_base) Y X_new (pre_new)
# ============================================================

# --- X_existing: use the base preprocessor ---
X_train_existing = pre_base.transform(X_train_existing_raw)
X_val_existing   = pre_base.transform(X_val_existing_raw)
X_test_existing  = pre_base.transform(X_test_existing_raw)

print("X_train_existing shape:", X_train_existing.shape)
print("X_val_existing shape:",   X_val_existing.shape)
print("X_test_existing shape:",  X_test_existing.shape)

assert X_train_existing.shape[1] == input_dim_base, "Input dimension does not match the base model"

# --- X_new: preprocessor new (fit solo on train) ---
num_cols_new = X_train_new_raw.select_dtypes(
    include=["number", "float", "int", "int64", "float64"]
).columns.tolist()
cat_cols_new = [c for c in X_train_new_raw.columns if c not in num_cols_new]

print("X_new numeric:", num_cols_new)
print("X_new categorical:", cat_cols_new)

pre_new = ColumnTransformer(
    transformers=[
        ("cat_new",
         OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         cat_cols_new),
        ("num_new",
         StandardScaler(),
         num_cols_new),
    ]
)

X_train_new = pre_new.fit_transform(X_train_new_raw)
X_val_new   = pre_new.transform(X_val_new_raw)
X_test_new  = pre_new.transform(X_test_new_raw)

print("X_train_new shape:", X_train_new.shape)
print("X_val_new shape:",   X_val_new.shape)
print("X_test_new shape:",  X_test_new.shape)


X_train_existing shape: (88, 8)
X_val_existing shape: (19, 8)
X_test_existing shape: (19, 8)
X_new numeric: ['Previous_therapy', 'Only_Chemotherapy', 'Chemotherapy_Irradiation']
X_new categorical: ['Graft_manipulation', 'Degree_of_kinship']
X_train_new shape: (88, 12)
X_val_new shape: (19, 12)
X_test_new shape: (19, 12)


#F1 CALLBACK FOR MULTI-INPUT


In [11]:
# ============================================================
# 7) F1 CALLBACK FOR MULTI-INPUT
# ============================================================
class ValF1MultiInput(Callback):
    def __init__(self, val_data, threshold=0.5, average='binary', verbose=False):
        super().__init__()
        (X_val_exist, X_val_new), y_val = val_data
        self.X_val_exist = X_val_exist
        self.X_val_new = X_val_new
        self.y_val = y_val
        self.threshold = threshold
        self.average = average
        self.verbose = verbose
        self.best = -1.0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_prob = self.model.predict(
            [self.X_val_exist, self.X_val_new],
            verbose=0
        ).ravel()
        y_pred = (y_prob >= self.threshold).astype(int)
        val_f1 = f1_score(self.y_val, y_pred, average=self.average)
        logs["val_f1"] = float(val_f1)
        if val_f1 > self.best:
            self.best = val_f1
        if self.verbose:
            print(f"\nEpoch {epoch+1}: val_f1={val_f1:.4f} (best={self.best:.4f})")


#DEFINE A MULTI-INPUT MODEL WITH A PRETRAINED ENCODER


In [12]:
# ============================================================
# 8) HYPERMODEL: build_transfer_model FOR THE TUNER
#
# ============================================================
# Encoder from base model frozen
encoder_model.trainable = False

input_dim_new = X_train_new.shape[1]

def build_transfer_model(hp: kt.HyperParameters):
    # Inputs
    in_exist = layers.Input(shape=(input_dim_base,), name="X_existing")
    in_new   = layers.Input(shape=(input_dim_new,),  name="X_new")

    # ---------------------------------------------------
    # 1) EXISTING branch: encoder pretrain (frozen)
    # ---------------------------------------------------
    z_exist = encoder_model(in_exist)  # output of the layer 'encoder'

    # ---------------------------------------------------
    # 2) NEW branch: number de layers + units tunables
    # ---------------------------------------------------
    # nº de layers ocultas on X_new: 0, 1 o 2
    n_layers_new = hp.Int("n_layers_new", min_value=0, max_value=2, step=1)
    dr_new = hp.Choice("dropout_new", values=[0.0, 0.2, 0.4])

    x_new = in_new
    for i in range(n_layers_new):
        units_i = hp.Int(f"new_units_{i+1}", min_value=8, max_value=32, step=8)
        x_new = layers.Dense(
            units_i,
            activation="relu",
            name=f"new_dense_{i+1}"
        )(x_new)
        if dr_new > 0:
            x_new = layers.Dropout(dr_new, name=f"new_dropout_{i+1}")(x_new)

    # If n_layers_new = 0, the new branch is simply the raw transformed input
    # (in this case without dropout).

    # ---------------------------------------------------
    # 3) Branch fusion
    # ---------------------------------------------------
    merged = layers.Concatenate(name="merge_exist_new")([z_exist, x_new])

    # ---------------------------------------------------
    # 4) Final head: nº de layers + units tunables
    # ---------------------------------------------------
    # at least 1 layer in the head, up to 2
    n_layers_head = hp.Int("n_layers_head", min_value=1, max_value=2, step=1)
    dr_head = hp.Choice("dropout_head", values=[0.0, 0.2, 0.4])

    h = merged
    for j in range(n_layers_head):
        units_h = hp.Int(f"head_units_{j+1}", min_value=8, max_value=32, step=8)
        h = layers.Dense(
            units_h,
            activation="relu",
            name=f"head_dense_{j+1}"
        )(h)
        if dr_head > 0:
            h = layers.Dropout(dr_head, name=f"head_dropout_{j+1}")(h)

    # Output layer
    out = layers.Dense(1, activation="sigmoid", name="final_head")(h)

    model = models.Model(inputs=[in_exist, in_new],
                         outputs=out,
                         name="full_transfer_model")

    # ---------------------------------------------------
    # 5) Compilation
    # ---------------------------------------------------
    lr = hp.Choice("lr", values=[1e-3, 5e-4, 1e-4])
    opt = optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )
    return model



#TRAINING


In [13]:
# ============================================================
# 9) KERAS TUNER: RandomSearch FOR THE MULTI-INPUT MODEL
# ============================================================
val_f1_cb = ValF1MultiInput(
    val_data=(([X_val_existing, X_val_new]), y_val_new),
    threshold=0.5,
    average='binary',
    verbose=False
)

es_tuner = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

rlr_tuner = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-5,
    verbose=1
)

tuner_transfer = kt.RandomSearch(
    hypermodel=build_transfer_model,
    objective=kt.Objective("val_f1", direction="max"),
    max_trials=MAX_TRIALS_TRANSFER,
    executions_per_trial=1,
    overwrite=False,
    directory="tuner_transfer",
    project_name="multi_input_transfer"
)

tuner_transfer.search(
    [X_train_existing, X_train_new],
    y_train_new,
    validation_data=([X_val_existing, X_val_new], y_val_new),
    epochs=EPOCHS_TRANSFER,
    batch_size=BATCH_SIZE_TRANSFER,
    callbacks=[val_f1_cb, es_tuner, rlr_tuner],
    verbose=2
)

print("Tuner search for the multi-input model completed.")


Trial 10 Complete [00h 00m 10s]
val_f1: 0.6666666666666666

Best val_f1 So Far: 0.7857142857142857
Total elapsed time: 00h 02m 39s
Tuner search for the multi-input model completed.


##BEST HYPERPARAMETERS AND BEST MODEL


In [14]:
# ============================================================
# 10) Best HYPERPARAMETERS Y BEST MODEL
# ============================================================
best_hp_transfer = tuner_transfer.get_best_hyperparameters(1)[0]
print("Best hyperparameters (model multi-input):")
for name in best_hp_transfer.values:
    print(f"  {name}: {best_hp_transfer.get(name)}")

best_model_transfer = tuner_transfer.get_best_models(1)[0]
best_model_transfer.summary()


Best hyperparameters (model multi-input):
  n_layers_new: 2
  dropout_new: 0.2
  n_layers_head: 1
  dropout_head: 0.4
  head_units_1: 32
  lr: 0.0005
  new_units_1: 8
  head_units_2: 16
  new_units_2: 8


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "full_transfer_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ X_new (InputLayer)  │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ new_dense_1 (Dense) │ (None, 8)         │        104 │ X_new[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ new_dropout_1       │ (None, 8)         │          0 │ new_dense_1[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ X_existing          │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ new_dense_2 (Dense) │ (None, 8)         │         72 │ new_dropout_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gvhd_encoder        │ (None, 8)         │         72 │ X_existing[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ new_dropout_2       │ (None, 8)         │          0 │ new_dense_2[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge_exist_new     │ (None, 16)        │          0 │ gvhd_encoder[11]… │
│ (Concatenate)       │                   │            │ new_dropout_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense_1        │ (None, 32)        │        544 │ merge_exist_new[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dropout_1      │ (None, 32)        │          0 │ head_dense_1[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_head (Dense)  │ (None, 1)         │         33 │ head_dropout_1[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 825 (3.22 KB)

 Trainable params: 753 (2.94 KB)

 Non-trainable params: 72 (288.00 B)

##SEARCH DEL BEST THRESHOLD EN VALIDATION (F1) Y EVALUATION


In [15]:
# ============================================================
# 11) THRESHOLD SEARCH ON VALIDATION AND EVALUATION ON TRAIN/VAL/TEST
# ============================================================
def find_best_threshold_on_val(model, X_exist_val, X_new_val, y_val, thr_grid=None):
    if thr_grid is None:
        thr_grid = np.linspace(0.1, 0.9, 17)

    y_prob = model.predict([X_exist_val, X_new_val], verbose=0).ravel()
    f1s = []
    for t in thr_grid:
        y_pred = (y_prob >= t).astype(int)
        f1s.append(f1_score(y_val, y_pred))

    best_idx = int(np.argmax(f1s))
    best_t = float(thr_grid[best_idx])
    best_f1 = float(f1s[best_idx])

    print("Threshold search on VALIDATION:")
    for t, f in zip(thr_grid, f1s):
        print(f"  thr={t:.2f} -> F1={f:.3f}")
    print(f"\nBest threshold on VAL: {best_t:.3f} with F1_val={best_f1:.3f}")

    return best_t, best_f1


def eval_split_multi(name, X_exist, X_new, y, model, thr):
    y_prob = model.predict([X_exist, X_new], verbose=0).ravel()
    y_pred = (y_prob >= thr).astype(int)

    print(f"\n==== {name.upper()} ====")
    print(f"Threshold = {thr:.3f}")
    print("ROC-AUC:", round(roc_auc_score(y, y_prob), 3))
    print("PR-AUC :", round(average_precision_score(y, y_prob), 3))
    print("Balanced accuracy:", round(balanced_accuracy_score(y, y_pred), 3))

    print("\nClassification report:")
    print(classification_report(y, y_pred, digits=3))

    cm = confusion_matrix(y, y_pred)
    print("Confusion matrix (rows = true, cols = predicted):")
    print(cm)

# 1) Best threshold using ONLY validation
best_t, best_f1_val = find_best_threshold_on_val(
    best_model_transfer,
    X_val_existing, X_val_new,
    y_val_new
)

# 2) Evaluate on train / val / test with that threshold
eval_split_multi("train", X_train_existing, X_train_new, y_train_new, best_model_transfer, thr=best_t)
eval_split_multi("val",   X_val_existing,   X_val_new,   y_val_new,   best_model_transfer, thr=best_t)
eval_split_multi("test",  X_test_existing,  X_test_new,  y_test,      best_model_transfer, thr=best_t)


Threshold search on VALIDATION:
  thr=0.10 -> F1=0.733
  thr=0.15 -> F1=0.733
  thr=0.20 -> F1=0.733
  thr=0.25 -> F1=0.733
  thr=0.30 -> F1=0.733
  thr=0.35 -> F1=0.733
  thr=0.40 -> F1=0.733
  thr=0.45 -> F1=0.733
  thr=0.50 -> F1=0.786
  thr=0.55 -> F1=0.133
  thr=0.60 -> F1=0.000
  thr=0.65 -> F1=0.000
  thr=0.70 -> F1=0.000
  thr=0.75 -> F1=0.000
  thr=0.80 -> F1=0.000
  thr=0.85 -> F1=0.000
  thr=0.90 -> F1=0.000

Best threshold on VAL: 0.500 with F1_val=0.786

==== TRAIN ====
Threshold = 0.500
ROC-AUC: 0.531
PR-AUC : 0.656
Balanced accuracy: 0.477

Classification report:
              precision    recall  f1-score   support

           0      0.350     0.200     0.255        35
           1      0.588     0.755     0.661        53

    accuracy                          0.534        88
   macro avg      0.469     0.477     0.458        88
weighted avg      0.493     0.534     0.499        88

Confusion matrix (rows = true, cols = predicted):
[[ 7 28]
 [13 40]]

==== VAL ====
Thre

In [16]:
import os
import json
import joblib

# Folder for the transfer model
TRANSFER_OUT_DIR = "outputs_nn_transfer"
os.makedirs(TRANSFER_OUT_DIR, exist_ok=True)

# Artifact paths
MODEL_TRANSFER_PATH   = os.path.join(TRANSFER_OUT_DIR, "multi_input_transfer_best.keras")
PRE_NEW_PATH          = os.path.join(TRANSFER_OUT_DIR, "pre_new.joblib")
THRESHOLD_TRANSFER    = os.path.join(TRANSFER_OUT_DIR, "threshold_transfer.json")
META_TRANSFER_PATH    = os.path.join(TRANSFER_OUT_DIR, "meta_transfer.json")

print("Folder transfer:", TRANSFER_OUT_DIR)
print("Transfer model:", MODEL_TRANSFER_PATH)
print("Pre_new:", PRE_NEW_PATH)
print("Threshold:", THRESHOLD_TRANSFER)
print("Meta transfer:", META_TRANSFER_PATH)


Folder transfer: outputs_nn_transfer
Transfer model: outputs_nn_transfer/multi_input_transfer_best.keras
Pre_new: outputs_nn_transfer/pre_new.joblib
Threshold: outputs_nn_transfer/threshold_transfer.json
Meta transfer: outputs_nn_transfer/meta_transfer.json


In [17]:
# ===== Save the BEST MODEL from the tuner =====
best_model_transfer.save(MODEL_TRANSFER_PATH, overwrite=True)
print("Multi-input model (best from tuner) saved to:", MODEL_TRANSFER_PATH)


Multi-input model (best from tuner) saved to: outputs_nn_transfer/multi_input_transfer_best.keras


In [18]:
# ===== Save preprocessor for the new features (X_new) =====
joblib.dump(pre_new, PRE_NEW_PATH)
print("Preprocessor of X_new saved to:", PRE_NEW_PATH)


Preprocessor of X_new saved to: outputs_nn_transfer/pre_new.joblib


In [19]:
# ===== Save information about the optimal THRESHOLD =====
thr_info = {
    "best_threshold": float(best_t),
    "best_f1_val": float(best_f1_val)
}
with open(THRESHOLD_TRANSFER, "w") as f:
    json.dump(thr_info, f, indent=2)

print("Optimal threshold saved to:", THRESHOLD_TRANSFER)


Optimal threshold saved to: outputs_nn_transfer/threshold_transfer.json


In [20]:
# ===== Save META from model transfer =====

meta_transfer = {
    "target": TARGET_NEW,
    "existing_cols_in_new": existing_cols_in_new,   # columns that go into the encoder
    "feature_cols_new": feature_cols_new,           # columns that go through X_new
    "num_cols_new": num_cols_new,
    "cat_cols_new": cat_cols_new,
    "input_dim_existing": int(X_train_existing.shape[1]),
    "input_dim_new": int(X_train_new.shape[1]),
    "test_size_global": TEST_SIZE_GLOBAL,
    "random_state_new": RANDOM_STATE_NEW,
    "n_train": int(len(y_train_new)),
    "n_val": int(len(y_val_new)),
    "n_test": int(len(y_test)),
    "best_threshold": float(best_t),
    "best_f1_val": float(best_f1_val),
    "best_hyperparameters": {name: best_hp_transfer.get(name)
                             for name in best_hp_transfer.values},
    "model_transfer_path": MODEL_TRANSFER_PATH,
    "pre_base_path": PREPROCESSOR_PATH,  # from the base model
    "pre_new_path": PRE_NEW_PATH,
    "threshold_path": THRESHOLD_TRANSFER
}

with open(META_TRANSFER_PATH, "w") as f:
    json.dump(meta_transfer, f, indent=2)

print("Transfer model metadata saved to:", META_TRANSFER_PATH)


Transfer model metadata saved to: outputs_nn_transfer/meta_transfer.json
